In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

import torch.nn.functional as F
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from tqdm import tqdm

In [14]:
# ==========================================
# 1. Data Loading & Preprocessing
# ==========================================
DATASET_PATH = "./fonts_dataset"

images = []
labels = []

# Load image files and extract labels from filenames
filenames = sorted(os.listdir(DATASET_PATH))
for filename in filenames:
    if filename.endswith(".png"):
        # Assuming filename format
        parts = filename.replace(".png", "").split("_")
        font_label = parts[1]

        img_path = os.path.join(DATASET_PATH, filename)
        img = Image.open(img_path).convert("L")  # Convert to grayscale
        img_array = np.array(img)

        images.append(img_array)
        labels.append(font_label)

X = np.array(images)  
y = np.array(labels)

print(f"Total loaded images: {len(X)}")

# Encode string labels to integer indices
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)

# Split dataset into training and validation sets (80:20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)
print(f"Train samples: {len(X_train)}, Validation samples: {len(X_val)}")


Total loaded images: 25000
Train samples: 20000, Validation samples: 5000


In [16]:
# ==========================================
# 2. Custom Dataset & Data Augmentation
# ==========================================
class AddGaussianNoise(object):
    """Custom transform to add Gaussian noise to images."""
    def __init__(self, mean=0., std=0.05):
        self.mean = mean
        self.std = std
        
    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std + self.mean
    
    def __repr__(self):
        return f'{self.__class__.__name__}(mean={self.mean}, std={self.std})'


class FontDataset(Dataset):
    """Custom PyTorch Dataset for Font images."""
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        image = self.X[idx]  # Shape: (H, W)
        label = self.y[idx]

        if self.transform:
            image = self.transform(image)
        else:
            # Default scaling if no transform is provided
            image = torch.tensor(image, dtype=torch.float32).unsqueeze(0) / 255.0

        return image, torch.tensor(label, dtype=torch.long)


# Define transformation pipelines for train and validation
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    AddGaussianNoise(mean=0., std=0.05),
    transforms.Normalize((0.5,), (0.5,))
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Create DataLoaders
batch_size = 64
train_dataset = FontDataset(X_train, y_train, transform=train_transform)
val_dataset = FontDataset(X_val, y_val, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)



In [17]:
# ==========================================
# 3. Model Architecture (CNN)
# ==========================================
class FontCNN(nn.Module):
    """CNN architecture for Font Classification."""
    def __init__(self, num_classes):
        super(FontCNN, self).__init__()
        
        # Feature extraction layers
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Global Average Pooling ensures fixed output size regardless of input spatial dimensions
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1) # Flatten (B, C, 1, 1) to (B, C)
        x = self.classifier(x)
        return x



In [ ]:
# ==========================================
# 4. Training Setup & Loop
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = FontCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    

    train_preds = [] 
    train_targets = [] 
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        

        train_preds.extend(predicted.cpu().numpy())
        train_targets.extend(labels.cpu().numpy())
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total * 100
    

    epoch_f1 = f1_score(train_targets, train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    

    val_preds = []
    val_targets = []
    
    with torch.no_grad():
        for val_images, val_labels in val_loader:
            val_images, val_labels = val_images.to(device), val_labels.to(device)
            val_outputs = model(val_images)
            v_loss = criterion(val_outputs, val_labels)
            
            val_loss += v_loss.item() * val_images.size(0)
            _, val_predicted = torch.max(val_outputs, 1)
            val_total += val_labels.size(0)
            val_correct += (val_predicted == val_labels).sum().item()
            
         
            val_preds.extend(val_predicted.cpu().numpy())
            val_targets.extend(val_labels.cpu().numpy())
            
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total * 100
    

    epoch_val_f1 = f1_score(val_targets, val_preds, average='macro')
    

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%, Train F1: {epoch_f1:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%, Val F1: {epoch_val_f1:.4f}")

Using device: cpu


KeyboardInterrupt: 